UMAP, PCA i TSNE 2d/3d na mnist i vis interactive np w tensorboard albo innym toolu

In [13]:
import umap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [10]:
digits = load_digits()
X, y = digits.data, digits.target # (1797, 64)
np.random.seed(42)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

methods = {}

pca_2d = PCA(n_components=2)
methods['PCA 2D'] = {'embed': pca_2d.fit_transform(X_scaled)}

pca_3d = PCA(n_components=3)
methods['PCA 3D'] = {'embed': pca_3d.fit_transform(X_scaled)}

tsne_2d = TSNE(n_components=2, perplexity=30, max_iter=1000, init='pca')
methods['t-SNE 2D'] = {'embed': tsne_2d.fit_transform(X_scaled)}

tsne_3d = TSNE(n_components=3, perplexity=30, max_iter=1000, init='pca')
methods['t-SNE 3D'] = {'embed': tsne_3d.fit_transform(X_scaled)}

umap_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1)
methods['UMAP 2D'] = {'embed': umap_2d.fit_transform(X_scaled)}

umap_3d = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1)
methods['UMAP 3D'] = {'embed': umap_3d.fit_transform(X_scaled)}

In [15]:
COLORS = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#a9a9a9"
]
labels = [str(i) for i in range(10)]

specs = [
    [{"type": "scatter"}, {"type": "scatter"}, {"type": "scatter"}],
    [{"type": "scatter3d"}, {"type": "scatter3d"}, {"type": "scatter3d"}],
]

fig = make_subplots(
    rows=2, cols=3,
    specs=specs,
    subplot_titles=["PCA 2D", "t-SNE 2D", "UMAP 2D",
                    "PCA 3D", "t-SNE 3D", "UMAP 3D"],
    horizontal_spacing=0.05,
    vertical_spacing=0.08,
)

method_order_2d = ["PCA 2D", "t-SNE 2D", "UMAP 2D"]
method_order_3d = ["PCA 3D", "t-SNE 3D", "UMAP 3D"]

def add_traces(method_name, row, col, is_3d, show_legend):
    embed = methods[method_name]["embed"]
    for digit in range(10):
        mask = y == digit
        if is_3d:
            trace = go.Scatter3d(
                x=embed[mask, 0], y=embed[mask, 1], z=embed[mask, 2],
                mode="markers",
                marker=dict(size=3, color=COLORS[digit], opacity=0.7),
                name=str(digit),
                legendgroup=str(digit),
                showlegend=show_legend,
                hovertemplate=f"Number {digit}<br>x: %{{x:.2f}}<br>y: %{{y:.2f}}<br>z: %{{z:.2f}}<extra></extra>",
            )
        else:
            trace = go.Scatter(
                x=embed[mask, 0], y=embed[mask, 1],
                mode="markers",
                marker=dict(size=5, color=COLORS[digit], opacity=0.7),
                name=str(digit),
                legendgroup=str(digit),
                showlegend=show_legend,
                hovertemplate=f"Number {digit}<br>x: %{{x:.2f}}<br>y: %{{y:.2f}}<extra></extra>",
            )
        fig.add_trace(trace, row=row, col=col)

for col, name in enumerate(method_order_2d, start=1):
    add_traces(name, row=1, col=col, is_3d=False, show_legend=(col == 1))

for col, name in enumerate(method_order_3d, start=1):
    add_traces(name, row=2, col=col, is_3d=True, show_legend=False)

fig.update_layout(
    title=dict(
        text="PCA / t-SNE / UMAP — MNIST Digits (1797, 64)",
        font=dict(size=18),
    ),
    legend=dict(
        title="Number",
        itemsizing="constant",
        tracegroupgap=4,
    ),
    height=900,
    template="plotly_white",
)

for col in range(1, 4):
    fig.update_xaxes(showticklabels=False, row=1, col=col)
    fig.update_yaxes(showticklabels=False, row=1, col=col)

fig.write_html("mnist_dim_reduction.html")
fig.show()